# [Problem Name]

> **Time-box:** 45–60 minutes. Identify the hardest read/write trade-off early and spend your time there.

## Core requirements

1. _Requirement one._
2. _Requirement two._
3. _Requirement three._

## Stretch goals

- _Stretch goal one._
- _Stretch goal two._
- _Stretch goal three._

## Things the interviewer will probe

- **Pagination:** offset vs cursor. What breaks at scale?
- **Consistency:** eventual vs strong. Where can you afford lag?
- **Indexing:** which queries need indexes? What does a full-table scan cost?
- **Denormalization:** when is duplicating data worth it?
- **Idempotency:** which endpoints must be safe to retry?

---
## Setup

In [20]:
import json
import sqlite3

import pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from IPython.display import display
from pydantic import BaseModel

## Schema

Edit the SQL and re-run this cell to get a fresh in-memory database.

In [21]:
SCHEMA = """
-- Example table of users
CREATE TABLE users (
    id   INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);
"""

conn = sqlite3.connect(":memory:", check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript(SCHEMA)

print("Tables:", [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()])

Tables: ['users']


## API

> After editing any cell below, re-run from **App** down through **Client**.

In [11]:
# ── App + models ──────────────────────────────────────────────────────────────
app = FastAPI(title="[Problem Name]")

# Create classes like this to define payload of HTTP request
class CreateUser(BaseModel):
    name: str

In [12]:
# ── Endpoints ─────────────────────────────────────────────────────────────────
@app.post("/users", status_code=201)
def create_user(payload: CreateUser):
    with conn:
        conn.execute("INSERT INTO users(name) VALUES (?)", (payload.name,))
    return {"status": "ok"}

In [13]:
# ── Client ────────────────────────────────────────────────────────────────────
client = TestClient(app, raise_server_exceptions=True)
print(client.get("/healthz").json())

{'detail': 'Not Found'}


## Helpers

In [14]:
def call(method: str, path: str, **kwargs):
    r = getattr(client, method)(path, **kwargs)
    body = r.json() if r.content else None
    print(f"{method.upper():6s} {path}  →  {r.status_code}")
    if body is not None:
        print(json.dumps(body, indent=2))
    return r


def df(table: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {table}", conn)


def query(sql: str, *params) -> pd.DataFrame:
    return pd.read_sql(sql, conn, params=list(params) if params else None)


def show_all():
    names = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]
    for name in names:
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"\n── {name} ({count} rows) ──")
        display(pd.read_sql(f"SELECT * FROM {name}", conn))

## Demo

In [15]:
# Call your endpoints here
call("post", "/users", json={"name": "Darius"})
df("users")

POST   /users  →  201
{
  "status": "ok"
}


,id,name
0,1,Darius


## All tables

In [16]:
show_all()


── users (1 rows) ──


,id,name
0,1,Darius
